# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****here
LANGSMITH_API_KEY=****3f9a
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [5]:
from langchain.chat_models import init_chat_model

# initialize a commercial chat model (e.g., "openai:gpt-5") or use a local
# model (e.g., "ollama:gpt-oss")
model = init_chat_model("ollama:gpt-oss:latest", temperature=0)

In [6]:
from langchain.agents import create_agent

# create your agent! Add the model object you just created, a prompt etc.
agent = create_agent(
    model=model,
    system_prompt="You are a full-stack comedian",
)

## No Streaming (invoke)

In [7]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

Why did the full‑stack developer break up with the database?

Because every time they tried to commit, the database said, “I’m not ready to be in a *transaction*—I need more *space*!”


## values
You have seen this streaming mode in our examples so far. 

In [8]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke
================================== Ai Message ==================================

Why don’t skeletons fight each other?  

They don’t have the guts.


## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [9]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

**A Day in the Sunlit Garden**

In the garden where the daisies dance,  
We gather round with a merry glance.  
The wind hums a tune so bright and clear,  
And every petal smiles back at us, dear.

We toss a feathered kite up high,  
It soars like a dragon in the sky.  
The children giggle, the parents grin,  
All worries drift away, let the fun begin.

A picnic blanket, crumbs of cake,  
We share stories for old and for the young’s sake.  
The laughter echoes, the world feels right,  
Under the golden, gentle light.

When the sun dips low, the stars appear,  
We whisper wishes, hold each other near.  
A family’s heart, a gentle song,  
Together we’ll keep this love lifelong.

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [10]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='b3860574-7215-4eae-8cb2-f06842b125a2')]})
('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='b3860574-7215-4eae-8cb2-f06842b125a2'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:latest', 'created_at': '2026-06-17T09:18:02.207600401Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2925026806, 'load_duration': None, 'prompt_eval_count': 128, 'prompt_eval_duration': None, 'eval_count': 34, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:latest', 'model_provider': 'ollama'}, id='lc_run--019ed4df-856f-7763-a12e-7774e58d9d9c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': '0e13229d-2e50-4cdd-9d3f-039fc93418ea', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 128, 'output_tokens': 34, 'to

In [11]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: SF')
('custom', 'Acquired data for city: SF')


## Try different modes on your own!
Modify the stream mode and the select to produce different results.

In [12]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])

Looking up data for city: SF
Acquired data for city: SF
